# M5 Forecasting — CPG Sales EDA

Exploratory data analysis on the Walmart M5 dataset (`sales_train_validation.csv`).

**Dataset:** 30,490 time series × 1,913 daily columns, covering 3 categories: FOODS, HOBBIES, HOUSEHOLD across 10 Walmart stores in CA, TX, WI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully.')

## 1. Load Data

In [ ]:
# Place sales_train_validation.csv in the same directory as this notebook,
# or update DATA_PATH accordingly.
DATA_PATH = 'sales_train_validation.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')

## 2. Basic Inspection

In [ ]:
df.head()

In [ ]:
df.info(verbose=False, show_counts=True)

In [ ]:
# Describe a sample of day-columns to keep output manageable
cols_to_describe = ['d_1', 'd_100', 'd_500', 'd_1000', 'd_1913']
df[cols_to_describe].describe().round(2)

## 3. Melt to Long Format — (item_id, store_id, date) × sales

The raw file has one row per product-store pair and one column per day (`d_1` ... `d_1913`).  
We melt it into a tidy long DataFrame: each row is one observation for one item-store on one day.

In [ ]:
id_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in df.columns if c.startswith('d_')]

long = df.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

# Add integer day number for sorting
long['day_num'] = long['d'].str.extract(r'(\d+)').astype(int)
long = long.sort_values(['item_id', 'store_id', 'day_num']).reset_index(drop=True)

print(f'Long DataFrame shape: {long.shape}')
long.head(10)

## 4. Confirm ~30,000 Unique (item, store) Series

In [ ]:
n_series = long.groupby(['item_id', 'store_id']).ngroups
print(f'Unique (item_id, store_id) series: {n_series:,}')
assert 29_000 <= n_series <= 31_000, f'Expected ~30,000 series, got {n_series}'
print('Confirmed ~30,000 series.')

## 5. Total Sales Per Category

In [ ]:
category_totals = long.groupby('cat_id')['sales'].sum().sort_values(ascending=False)
print('Total sales per category:')
print(category_totals.to_string())

category_totals.plot(
    kind='bar',
    title='Total Sales by Category (all stores, all time)',
    ylabel='Total Units Sold',
    xlabel='Category',
    rot=0,
    figsize=(7, 4),
    color=['#e07b39', '#3a86ff', '#43aa8b']
)
plt.tight_layout()
plt.show()

## 6. Daily Total Sales for FOODS

In [ ]:
foods_daily = (
    long[long['cat_id'] == 'FOODS']
    .groupby('day_num')['sales']
    .sum()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(
    foods_daily['day_num'],
    foods_daily['sales'],
    linewidth=0.8,
    color='#e07b39',
    alpha=0.85
)
ax.set(
    title='Daily Total Sales - FOODS Category (all stores)',
    xlabel='Day index (d_1 to d_1913)',
    ylabel='Total Units Sold'
)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('foods_daily_sales.png', dpi=150)
plt.show()
print('Plot saved to foods_daily_sales.png')